In [1]:
#!/usr/bin/env python
# coding: utf-8


import os
#Directory here
os.chdir('/Users/cherryleheu/Codes/NIDIS-Codes/H-RIP/Python')


import geopandas as gpd
import shapely 
from shapely.geometry import Point
import matplotlib.pyplot as plt
# from dms2dec.dms_convert import dms2dec
import numpy as np
import pandas as pd
import rasterio
# from rasterio.plot import show
from rasterstats import zonal_stats
from datetime import datetime, timedelta
import calendar
from dateutil.relativedelta import *
# from standard_precip import spi

In [ ]:
# Read the shapefile
gdf = gpd.read_file('/Users/cherryleheu/Codes/NIDIS-Codes/H-RIP/Python/shapefiles/RID.shp')
thisYear=(datetime.today().strftime("%Y"))
thisMonth=2
# yestYr, yestMonth, yestDay = int((datetime.today() + relativedelta(days=-1)).strftime("%Y")),int((datetime.today() + relativedelta(days=-1)).strftime("%m")),int((datetime.today() + relativedelta(days=-1)).strftime("%d"))
months = np.arange(1,13,1)

for index, row in gdf.iterrows():
    ranchid = row['Polygon']
    geom = row.geometry
    temp = pd.DataFrame({'Year': [],'Month':[],'tmean_f': []})
    for i in np.arange(1990,int(thisYear)+1):
        for j in months:
            if int(i) == int(thisYear) and int(j)==(thisMonth):
                break
            with rasterio.open(f'/Users/cherryleheu/Documents/HCDP/Data/monthly/tmean/tmean_{i}_{j:02d}.tif') as src:
                affine = src.transform
                array = src.read(1)
                df_zonal_stats = pd.DataFrame(zonal_stats(geom, array, affine=affine,nodata=-9999,stats = ['mean']))
            temp_f = 9/5*df_zonal_stats['mean'].iloc[0]+32
            new_row = pd.DataFrame({'Year':i,'Month':j,'tmean_f':temp_f},index=[0])
            temp=pd.concat([temp, new_row],ignore_index=True)
    temp['datetime']=pd.date_range('1/1/1990','2/1/2026',freq='MS')
    temp.to_csv(f'../RID/{ranchid}/{ranchid}_tmean.csv',index=False)

In [8]:
gdf = gpd.read_file('./shapefiles/RID.shp')
lastMonth = int((datetime.today() + relativedelta(months=-1)).strftime("%m"))
lastMonthYr = (datetime.today() + relativedelta(months=-1)).strftime("%Y")
lastMonthDays = calendar.monthrange(int(lastMonthYr), int(lastMonth))[1]

thisYear=(datetime.today().strftime("%Y"))
thisMonth=int((datetime.today().strftime("%m")))

for index, row in gdf.iterrows():
    ranchid = row['Polygon']
    geom = row['geometry']
    lastMonthDailyRF = []
    thisMonthDailyRF = []
    pd.options.display.float_format = '{:.6f}'.format
    for day in np.arange(1,lastMonthDays+1):
        with rasterio.open(f'./rain_daily_maps/{thisYear}_{lastMonth:02d}_{day:02d}.tif') as src:
            affine = src.transform
            array = src.read(1)
            df_zonal_stats = pd.DataFrame(zonal_stats(geom, array, affine=affine,nodata=src.nodata,stats = ['mean']))
        rf = (df_zonal_stats['mean'].iloc[0])/25.4
        row = {'Year': lastMonthYr, 'Month': lastMonth, 'Day':day, 'RF_in': rf}
        lastMonthDailyRF.append(row)
    df = pd.DataFrame(lastMonthDailyRF)
    df.to_csv(f'../RID/{ranchid}/{ranchid}_rf_daily_last_month.csv',index=False)

In [8]:
dir_path = '../RID'
count = 0
for path in os.listdir(dir_path):
    if os.path.isdir(os.path.join(dir_path, path)):
        count += 1

lastMonth = (datetime.today() + relativedelta(months=-1)).strftime("%m")
lastMonthYr = (datetime.today() + relativedelta(months=-1)).strftime("%Y")

datem = datetime.today().strftime("%m")
monthInd = -int(datem)+1

#Mean temperature download
url = 'https://ikeauth.its.hawaii.edu/files/v2/download/public/system/ikewai-annotated-data/HCDP/production/temperature/mean/month/statewide/data_map/'+lastMonthYr+'/temperature_mean_month_statewide_data_map_'+lastMonthYr+'_'+lastMonth+'.tif'

import requests 
r = requests.get(url)

file=f"./temp_monthly_maps/mean/t_month_mean_{lastMonthYr}_{lastMonth}.tif"
open(file, 'wb').write(r.content)
dir_path = '../RID'
count = 0
for path in os.listdir(dir_path):
    if os.path.isdir(os.path.join(dir_path, path)):
        count += 1


ranches = np.arange(1,count+1)

for r in ranches:
    ranchshp = gpd.read_file('./shapefiles/RID.shp',rows=slice(r-1, r))
    ranch = ranchshp['Polygon'].iloc[0]
    csv = pd.read_csv('../RID/'+ranch+'/'+ranch+'_tmean.csv')
    with rasterio.open(file) as src:
        affine = src.transform
        array = src.read(1)
        noData=src.nodata
        df_zonal_stats = pd.DataFrame(zonal_stats(ranchshp, array, affine=affine,nodata=noData,stats = ['mean']))
    temp_c= df_zonal_stats['mean'].iloc[0]
    temp_f = 9/5*temp_c+32
    new_row = pd.DataFrame({'Year':int(lastMonthYr),'Month':int(lastMonth),'Temp':temp_f},index=[0])
    csv=pd.concat([csv, new_row],ignore_index=True)
    csv['datetime']=pd.date_range('1/1/1990',lastMonth+'/01/'+lastMonthYr,freq='MS')
    csv.to_csv('../RID/'+ranch+'/'+ranch+'_tmean.csv', index=False)


In [7]:
csv

,Year,Month,Temp,datetime
0,1990,1.0,61.812706,1990-01-01
1,1990,2.0,59.398977,1990-02-01
2,1990,3.0,60.686279,1990-03-01
3,1990,4.0,63.124661,1990-04-01
4,1990,5.0,63.402151,1990-05-01
...,...,...,...,...
423,2025,4.0,63.962033,2025-04-01
424,2025,5.0,64.337188,2025-05-01
425,2025,6.0,65.982148,2025-06-01
426,2025,7.0,67.265449,2025-07-01
